<a href="https://colab.research.google.com/github/3160664535qqcom-ctrl/kangtai-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [ ]:
#trips.duration_min is the measure you average (AVG).
#bikes.bike_type is the first grouping variable (electric vs. standard).
#stations.zone is the second grouping variable. It’s the zone of the start station.

In [4]:
print("=== Summary of 'bikes' table ===")
display(tables['bikes'].info())
print("\nUnique bike types:")
display(tables['bikes']['bike_type'].value_counts())

print("\n=== Summary of 'stations' table ===")
display(tables['stations'].info())
print("\nUnique zones:")
display(tables['stations']['zone'].value_counts())

print("\n=== Summary of 'trips' table (bike_id overlap) ===")
print(f"Unique bike IDs in 'bikes' table: {tables['bikes']['bike_id'].nunique()}")
print(f"Unique bike IDs in 'trips' table: {tables['trips']['bike_id'].nunique()}")

=== Summary of 'bikes' table ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   bike_id        60 non-null     object
 1   bike_type      60 non-null     object
 2   purchase_year  60 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 1.5+ KB


None


Unique bike types:


,count
bike_type,
Standard,36
Electric,9
STANDARD,8
standard,3
Electric,2
electric,1
E-bike,1



=== Summary of 'stations' table ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_id    12 non-null     object 
 1   station_name  12 non-null     object 
 2   zone          12 non-null     object 
 3   capacity      11 non-null     float64
dtypes: float64(1), object(3)
memory usage: 516.0+ bytes


None


Unique zones:


,count
zone,
Centre,5
North,2
West,2
South,2
East,1



=== Summary of 'trips' table (bike_id overlap) ===
Unique bike IDs in 'bikes' table: 60
Unique bike IDs in 'trips' table: 60


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

### Cleaning Step: Standardize Bike Types and Check for Duplicates

1.  **Standardize `bike_type`**: We will map variations to either `Standard` or `Electric`.
2.  **Duplicate check**: We will verify if there are duplicate records in `bikes` (`bike_id`) or `trips` (`trip_id`).

In [5]:
# 1. Inspect unique values of bike_type before cleaning
bikes_df = tables['bikes'].copy()
print("Unique bike types before cleaning:", bikes_df['bike_type'].unique())

# Standardize bike_type into 'Electric' or 'Standard'
def standardize_type(btype):
    btype_str = str(btype).strip().lower()
    if 'elec' in btype_str or 'e-' in btype_str or 'ebike' in btype_str:
        return 'Electric'
    else:
        return 'Standard'

bikes_df['bike_type'] = bikes_df['bike_type'].apply(standardize_type)
print("Unique bike types after cleaning:", bikes_df['bike_type'].value_counts())

# Save the cleaned dataframe back to tables
tables['bikes'] = bikes_df
con.register('bikes', bikes_df) # update DuckDB

Unique bike types before cleaning: ['Standard' 'Electric ' 'electric' 'STANDARD' 'standard' 'Electric'
 'E-bike']
Unique bike types after cleaning: bike_type
Standard    47
Electric    13
Name: count, dtype: int64


In [7]:
# 2. Check for duplicate IDs in both tables
bikes_duplicates = tables['bikes']['bike_id'].duplicated().sum()
trips_duplicates = tables['trips']['trip_id'].duplicated().sum()

print(f"Number of duplicate bike_id in bikes table: {bikes_duplicates}")
print(f"Number of duplicate trip_id in trips table: {trips_duplicates}")

if trips_duplicates > 0:
    print("\nPreview of duplicate trip records:")
    # Corrected 'false' to 'False'
    duplicates_df = tables['trips'][tables['trips']['trip_id'].duplicated(keep=False)]
    display(duplicates_df.sort_values(by='trip_id'))

Number of duplicate bike_id in bikes table: 0
Number of duplicate trip_id in trips table: 12

Preview of duplicate trip records:


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member


In [9]:
trips_df = tables['trips'].copy()

# Identify duplicate trip_ids
dup_mask = trips_df.duplicated(subset=['trip_id'], keep=False)
dup_ids = trips_df.loc[dup_mask, 'trip_id'].unique()

rows_to_drop = []

for tid in dup_ids:
    sub = trips_df[trips_df['trip_id'] == tid]
    # If all columns (especially start_station_id and end_station_id) are identical, flag the rest for dropping
    if len(sub[['start_station_id', 'end_station_id']].drop_duplicates()) == 1:
        # Mark duplicates beyond the first row for deletion
        indices = sub.index.tolist()
        rows_to_drop.extend(indices[1:])
    else:
        # If start and end stations are not the same, assign a new unique trip_id to the secondary records
        indices = sub.index.tolist()
        for i, idx in enumerate(indices[1:], start=1):
            trips_df.at[idx, 'trip_id'] = f"{tid}_gen_{i}"

# Drop identical duplicates
if rows_to_drop:
    trips_df = trips_df.drop(rows_to_drop)

print(f"Cleaned trips table shape: {trips_df.shape}")
print(f"Remaining duplicate trip_ids: {trips_df['trip_id'].duplicated().sum()}")

# Update tables dictionary and DuckDB
tables['trips'] = trips_df
con.register('trips', trips_df)

Cleaned trips table shape: (600, 7)
Remaining duplicate trip_ids: 0


## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

In [10]:
import pandas as pd

# 1. Merge trips with bikes on bike_id to get bike_type
m_df = tables['trips'].merge(tables['bikes'][['bike_id', 'bike_type']], on='bike_id', how='inner')

# 2. Merge with stations on start_station_id to get the starting station's zone
m_df = m_df.merge(tables['stations'][['station_id', 'zone']], left_on='start_station_id', right_on='station_id', how='inner')

# 3. Perform GroupBy to find the count and average duration
result_df = m_df.groupby(['bike_type', 'zone']).agg(
    n_trips=('duration_min', 'count'),
    avg_duration_min=('duration_min', 'mean')
).reset_index()

# Display the final summary
display(result_df)

,bike_type,zone,n_trips,avg_duration_min
0,Electric,Centre,81,14.506173
1,Electric,East,11,15.727273
2,Electric,North,23,15.956522
3,Electric,South,24,13.416667
4,Electric,West,22,15.681818
5,Standard,Centre,217,38.658986
6,Standard,East,20,19.500000
7,Standard,North,39,20.358974
8,Standard,South,60,43.150000
9,Standard,West,73,19.821918


## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

give me a summary of the tables like bike type, bike id and zone in codes,
please help me clean this table. to standarize the bike type into 2 category, standard and electric. like E-bike, ELETRIC shall put into standard. count if the bike id trip id is repeated,
plese check the start_station_id and end_station_id to see if the duplicated trip_id are the same, if they are the same, delete one of them, if they are not, produce a new id
please calculate the average trip duration by bike type and the zone of the start station. use groupby in bike_type and zone, start station to find out if there is a big difference in standard and electric.

## 6. What I could not verify

*What remains unchecked? What would you do next?*